In [ ]:
"""
Inventário de Estrutura dos Arquivos BPS
TCC — Business Intelligence aplicado ao Banco de Preços em Saúde
Mônica Anatália — POLI USP PRO

Objetivo:
    Ler todos os arquivos BPS (.xls e .xlsx) de uma pasta e gerar
    uma tabela onde cada linha é um arquivo e cada coluna indica
    se aquele campo existe (✓) ou não existe (—) naquele arquivo.

Como usar no Google Colab:
    1. Faça upload dos arquivos BPS para uma pasta no Google Drive
    2. Monte o Drive:
         from google.colab import drive
         drive.mount('/content/drive')
    3. Ajuste a variável PASTA_BPS com o caminho correto
    4. Execute o script
"""

import os
import pandas as pd
from pathlib import Path



In [ ]:
# ─────────────────────────────────────────────
# CONFIGURAÇÃO — ajuste conforme seu ambiente
# ─────────────────────────────────────────────

# No Colab com Google Drive, use algo como:
PASTA_BPS = Path(PASTA_DADOS) / 'Base Tratamento Minimo'
ARQUIVO_SAIDA = 'inventario_estrutura_bps.xlsx'


In [ ]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────────
# FUNÇÕES AUXILIARES
# ─────────────────────────────────────────────

def detectar_header(caminho: str) -> int:
    """
    Arquivos BPS antigos (.xls) têm uma linha de aviso antes do cabeçalho.
    Esta função detecta automaticamente em qual linha está o cabeçalho real.
    """
    df_raw = pd.read_excel(caminho, header=None, nrows=5)
    for i, row in df_raw.iterrows():
        valores = [str(v) for v in row.values if pd.notna(v)]
        # Linha de cabeçalho real tem múltiplos valores não-nulos
        if len(valores) >= 3:
            return i
    return 0


def obter_colunas(caminho: str) -> list:
    """Retorna a lista de colunas de um arquivo BPS."""
    header_row = detectar_header(caminho)
    df = pd.read_excel(caminho, header=header_row, nrows=0)
    return list(df.columns)


def extrair_ano(nome_arquivo: str) -> str:
    """Extrai o ano do nome do arquivo (ex: BPS_2025.xlsx → 2025)."""
    import re
    match = re.search(r'(\d{4})', nome_arquivo)
    return match.group(1) if match else nome_arquivo



In [ ]:
# ─────────────────────────────────────────────
# EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────

def gerar_inventario(pasta: str, saida: str):
    pasta = Path(pasta)

    # Coleta todos os arquivos BPS
    arquivos = sorted([
        f for f in pasta.iterdir()
        if f.suffix.lower() in ('.xls', '.xlsx')
        and 'BPS' in f.name.upper()
        and 'inventario' not in f.name.lower()
    ])

    if not arquivos:
        print(f"Nenhum arquivo BPS encontrado em: {pasta.resolve()}")
        return

    print(f"Arquivos encontrados: {len(arquivos)}")
    for f in arquivos:
        print(f"  • {f.name}")

    # Mapeia arquivo → colunas
    mapa = {}
    for arq in arquivos:
        try:
            colunas = obter_colunas(str(arq))
            mapa[arq.name] = colunas
            print(f"  ✓ {arq.name}: {len(colunas)} colunas")
        except Exception as e:
            mapa[arq.name] = []
            print(f"  ✗ {arq.name}: ERRO — {e}")

    # Universo de colunas únicas (ordem de aparição)
    todas_colunas = []
    for cols in mapa.values():
        for c in cols:
            if c not in todas_colunas:
                todas_colunas.append(c)

    # Monta a tabela de inventário
    linhas = []
    for nome_arq, colunas in mapa.items():
        linha = {'Arquivo': nome_arq, 'Ano': extrair_ano(nome_arq), 'Qtd Colunas': len(colunas)}
        for col in todas_colunas:
            linha[col] = '✓' if col in colunas else '—'
        linhas.append(linha)

    df_inventario = pd.DataFrame(linhas)
    df_inventario = df_inventario.sort_values('Ano').reset_index(drop=True)


In [ ]:
def gerar_inventario(pasta: str, saida: str):
    pasta = Path(pasta)

    # Coleta todos os arquivos BPS
    arquivos = sorted([
        f for f in pasta.iterdir()
        if f.suffix.lower() in ('.xls', '.xlsx')
        and 'BPS' in f.name.upper()
        and 'inventario' not in f.name.lower()
    ])

    if not arquivos:
        print(f"Nenhum arquivo BPS encontrado em: {pasta.resolve()}")
        return

    print(f"Arquivos encontrados: {len(arquivos)}")
    for f in arquivos:
        print(f"  • {f.name}")

    # Mapeia arquivo → colunas
    mapa = {}
    for arq in arquivos:
        try:
            colunas = obter_colunas(str(arq))
            mapa[arq.name] = colunas
            print(f"  ✓ {arq.name}: {len(colunas)} colunas")
        except Exception as e:
            mapa[arq.name] = []
            print(f"  ✗ {arq.name}: ERRO — {e}")

    # Universo de colunas únicas (ordem de aparição)
    todas_colunas = []
    for cols in mapa.values():
        for c in cols:
            if c not in todas_colunas:
                todas_colunas.append(c)

    # Monta a tabela de inventário
    linhas = []
    for nome_arq, colunas in mapa.items():
        linha = {'Arquivo': nome_arq, 'Ano': extrair_ano(nome_arq), 'Qtd Colunas': len(colunas)}
        for col in todas_colunas:
            linha[col] = '✓' if col in colunas else '—'
        linhas.append(linha)

    df_inventario = pd.DataFrame(linhas)
    df_inventario = df_inventario.sort_values('Ano').reset_index(drop=True)

    # ── Exporta para Excel ──────────────────────
    caminho_saida = pasta / saida
    with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
        df_inventario.to_excel(writer, sheet_name='Inventário', index=False)

        ws = writer.sheets['Inventário']

        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        # Cores
        fill_header   = PatternFill('solid', fgColor='1F4E79')   # azul escuro
        fill_presente = PatternFill('solid', fgColor='C6EFCE')   # verde claro
        fill_ausente  = PatternFill('solid', fgColor='FFCCCC')   # vermelho claro
        fill_meta     = PatternFill('solid', fgColor='BDD7EE')   # azul claro (metadados)

        fonte_header  = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        fonte_dados   = Font(name='Arial', size=9)
        borda_fina    = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'),  bottom=Side(style='thin')
        )

        n_cols = df_inventario.shape[1]
        n_rows = df_inventario.shape[0]

        for col_idx in range(1, n_cols + 1):
            letra = get_column_letter(col_idx)
            header_cell = ws[f'{letra}1']

            # Formata cabeçalho
            header_cell.fill      = fill_header
            header_cell.font      = fonte_header
            header_cell.alignment = Alignment(horizontal='center', wrap_text=True)
            header_cell.border    = borda_fina

            # Largura das colunas
            col_name = df_inventario.columns[col_idx - 1]
            if col_name in ('Arquivo', 'Ano', 'Qtd Colunas'):
                ws.column_dimensions[letra].width = 22
            else:
                ws.column_dimensions[letra].width = 28

            # Formata células de dados
            for row_idx in range(2, n_rows + 2):
                cell = ws[f'{letra}{row_idx}']
                valor = cell.value
                cell.font      = fonte_dados
                cell.alignment = Alignment(horizontal='center')
                cell.border    = borda_fina

                if col_name in ('Arquivo', 'Ano', 'Qtd Colunas'):
                    cell.fill = fill_meta
                elif valor == '✓':
                    cell.fill = fill_presente
                else:
                    cell.fill = fill_ausente

        # Congela primeira linha e primeiras 3 colunas
        ws.freeze_panes = 'D2'

    print(f"\nInventário salvo em: {caminho_saida.resolve()}")
    print(f"Total: {len(arquivos)} arquivos | {len(todas_colunas)} colunas únicas")
    return df_inventario

In [ ]:
# ─────────────────────────────────────────────
# EXECUTA
# ─────────────────────────────────────────────
df = gerar_inventario(PASTA_BPS, ARQUIVO_SAIDA)
if df is not None:
    print("\nPrévia do resultado:")
    print(df[['Arquivo', 'Ano', 'Qtd Colunas']].to_string(index=False))